# Ultramarathon Performance Visualizations — Part C
**Group 17 — Luiz Samelo, Maximilian Staudacher, Rayudu Muralikrishna**  
*Information Visualization (VU 2.0), TU Wien, 2026*

This notebook covers:
- **Viz 5**: Age vs. speed scatter by gender (all distances, all years)
- **Viz 6**: Athlete career arc — individual trajectories (top 50 most active athletes)

## 0. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/TWO_CENTURIES_OF_UM_RACES.csv')
print('Shape:', df.shape)
df.head()

## 1. Data Cleaning & Preprocessing

In [ ]:
# Rename columns
df.columns = [
    'year', 'race_name', 'race_length', 'num_finishers',
    'athlete_performance', 'athlete_club', 'athlete_country',
    'athlete_year_of_birth', 'athlete_gender', 'athlete_age_category',
    'athlete_avg_speed', 'athlete_id'
]

# Convert types
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['athlete_avg_speed'] = pd.to_numeric(df['athlete_avg_speed'], errors='coerce')
df['athlete_year_of_birth'] = pd.to_numeric(df['athlete_year_of_birth'], errors='coerce')

# Calculate athlete age at race
df['athlete_age'] = df['year'] - df['athlete_year_of_birth']

# Base clean filter
df_clean = df[
    (df['athlete_avg_speed'] > 0) &
    (df['athlete_avg_speed'] < 30) &  # remove outliers
    (df['athlete_age'] >= 18) &
    (df['athlete_age'] <= 80) &
    (df['athlete_gender'].isin(['M', 'F'])) &
    (df['year'] >= 1970) &
    (df['year'] <= 2022)
].copy()

print('Cleaned shape:', df_clean.shape)

---
## Viz 5 — Age vs. Speed Scatter by Gender
**Research angle**: How does age relate to performance across genders in ultra-marathon running?

In [ ]:
# Sample for performance (scatter with millions of points is slow)
# Take a representative sample of 50,000 records
df_scatter = df_clean.sample(n=min(50000, len(df_clean)), random_state=42).copy()

# Map gender labels
df_scatter['Gender'] = df_scatter['athlete_gender'].map({'M': 'Male', 'F': 'Female'})

# Round age to integer
df_scatter['athlete_age'] = df_scatter['athlete_age'].astype(int)

print('Scatter sample size:', df_scatter.shape[0])
df_scatter[['athlete_age', 'athlete_avg_speed', 'Gender']].describe()

In [ ]:
# Compute mean speed per age per gender for trend line
age_trend = (
    df_clean
    .groupby(['athlete_age', 'athlete_gender'])
    .agg(mean_speed=('athlete_avg_speed', 'mean'),
         count=('athlete_avg_speed', 'count'))
    .reset_index()
)
age_trend = age_trend[age_trend['count'] >= 50]  # only reliable age groups
age_trend['Gender'] = age_trend['athlete_gender'].map({'M': 'Male', 'F': 'Female'})

age_trend.head()

In [ ]:
# Build scatter plot
fig5 = px.scatter(
    df_scatter,
    x='athlete_age',
    y='athlete_avg_speed',
    color='Gender',
    color_discrete_map={'Male': '#3A86FF', 'Female': '#FF006E'},
    opacity=0.15,
    labels={
        'athlete_age': 'Athlete Age',
        'athlete_avg_speed': 'Avg Speed (km/h)',
        'Gender': 'Gender'
    },
    title='Age vs. Average Speed by Gender — All Ultra-Marathon Distances (1970–2022)',
    hover_data={'athlete_age': True, 'athlete_avg_speed': ':.2f'}
)

# Overlay mean trend lines
for gender, color in [('Male', '#1a56cc'), ('Female', '#cc0052')]:
    trend = age_trend[age_trend['Gender'] == gender].sort_values('athlete_age')
    fig5.add_trace(go.Scatter(
        x=trend['athlete_age'],
        y=trend['mean_speed'],
        mode='lines',
        name=f'{gender} (mean)',
        line=dict(color=color, width=3),
        hovertemplate='Age: %{x}<br>Mean Speed: %{y:.2f} km/h<extra></extra>'
    ))

# Peak performance annotation
male_peak = age_trend[age_trend['Gender'] == 'Male'].loc[
    age_trend[age_trend['Gender'] == 'Male']['mean_speed'].idxmax()
]
female_peak = age_trend[age_trend['Gender'] == 'Female'].loc[
    age_trend[age_trend['Gender'] == 'Female']['mean_speed'].idxmax()
]

for peak, label, color in [
    (male_peak, 'Male peak', '#1a56cc'),
    (female_peak, 'Female peak', '#cc0052')
]:
    fig5.add_annotation(
        x=peak['athlete_age'],
        y=peak['mean_speed'],
        text=f"{label}<br>Age {int(peak['athlete_age'])}",
        showarrow=True,
        arrowhead=2,
        font=dict(size=11, color=color),
        arrowcolor=color,
        ax=40, ay=-40
    )

fig5.update_layout(
    height=560,
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=17,
    legend=dict(title='Gender', x=1.01, y=1),
    xaxis=dict(title='Athlete Age', range=[18, 80], showgrid=True, gridcolor='#f0f0f0'),
    yaxis=dict(title='Avg Speed (km/h)', showgrid=True, gridcolor='#f0f0f0'),
    hovermode='closest'
)

fig5.show()
fig5.write_html('../Viz/viz5_age_vs_speed_by_gender.html')
print('Saved to Viz/viz5_age_vs_speed_by_gender.html')

---
## Viz 6 — Athlete Career Arc (Individual Trajectories)
**Research angle**: How do individual athletes' speeds evolve across their career years in ultra-marathon running?

In [ ]:
# Find top 50 most active athletes (most race appearances)
athlete_counts = (
    df_clean
    .groupby('athlete_id')
    .agg(
        race_count=('year', 'count'),
        career_span=('year', lambda x: x.max() - x.min()),
        gender=('athlete_gender', 'first'),
        country=('athlete_country', 'first')
    )
    .reset_index()
)

# Keep athletes with at least 5 year career span and 10+ races
active_athletes = athlete_counts[
    (athlete_counts['career_span'] >= 5) &
    (athlete_counts['race_count'] >= 10)
].nlargest(50, 'race_count')

print('Top 50 athletes selected')
print(active_athletes[['athlete_id', 'race_count', 'career_span', 'gender']].head(10))

In [ ]:
# Get all race records for these athletes
top_athlete_ids = active_athletes['athlete_id'].tolist()
df_careers = df_clean[df_clean['athlete_id'].isin(top_athlete_ids)].copy()

# Compute yearly avg speed per athlete
career_yearly = (
    df_careers
    .groupby(['athlete_id', 'year'])
    .agg(
        avg_speed=('athlete_avg_speed', 'mean'),
        races=('athlete_avg_speed', 'count'),
        age=('athlete_age', 'mean'),
        gender=('athlete_gender', 'first'),
        country=('athlete_country', 'first')
    )
    .reset_index()
)

career_yearly['avg_speed'] = career_yearly['avg_speed'].round(2)
career_yearly['age'] = career_yearly['age'].round(1)
career_yearly['Gender'] = career_yearly['gender'].map({'M': 'Male', 'F': 'Female'})

# Create career year (years since first race)
career_yearly['first_year'] = career_yearly.groupby('athlete_id')['year'].transform('min')
career_yearly['career_year'] = career_yearly['year'] - career_yearly['first_year']

# Short athlete label
career_yearly['athlete_label'] = career_yearly['athlete_id'].astype(str).str[:8]

print('Career records:', career_yearly.shape)
career_yearly.head(10)

In [ ]:
# Build career arc chart
fig6 = px.line(
    career_yearly,
    x='career_year',
    y='avg_speed',
    color='Gender',
    line_group='athlete_id',
    color_discrete_map={'Male': '#3A86FF', 'Female': '#FF006E'},
    hover_data={
        'athlete_label': True,
        'avg_speed': ':.2f',
        'age': ':.0f',
        'races': True,
        'country': True,
        'career_year': True,
        'athlete_id': False,
        'Gender': False
    },
    labels={
        'career_year': 'Years into Career',
        'avg_speed': 'Avg Speed (km/h)',
        'athlete_label': 'Athlete ID',
        'age': 'Age',
        'races': 'Races that year',
        'country': 'Country'
    },
    title='Athlete Career Arcs — Top 50 Most Active Ultra-Marathon Runners',
    opacity=0.6
)

# Add mean career arc per gender
mean_arc = (
    career_yearly
    .groupby(['career_year', 'Gender'])
    .agg(mean_speed=('avg_speed', 'mean'))
    .reset_index()
)

for gender, color in [('Male', '#003f99'), ('Female', '#990033')]:
    arc = mean_arc[mean_arc['Gender'] == gender].sort_values('career_year')
    fig6.add_trace(go.Scatter(
        x=arc['career_year'],
        y=arc['mean_speed'],
        mode='lines',
        name=f'{gender} avg arc',
        line=dict(color=color, width=4, dash='dash'),
        hovertemplate='Career year: %{x}<br>Mean Speed: %{y:.2f} km/h<extra></extra>'
    ))

fig6.update_layout(
    height=580,
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=17,
    legend=dict(title='Gender', x=1.01, y=1),
    xaxis=dict(
        title='Years into Career',
        showgrid=True,
        gridcolor='#f0f0f0'
    ),
    yaxis=dict(
        title='Avg Speed (km/h)',
        showgrid=True,
        gridcolor='#f0f0f0'
    ),
    hovermode='closest'
)

fig6.show()
fig6.write_html('../Viz/viz6_athlete_career_arc.html')
print('Saved to Viz/viz6_athlete_career_arc.html')

---
## Summary

| Viz | Description | File saved |
|-----|-------------|------------|
| 5 | Age vs. speed scatter by gender | `Viz/viz5_age_vs_speed_by_gender.html` |
| 6 | Athlete career arc trajectories | `Viz/viz6_athlete_career_arc.html` |